## Topic 2: Chunk Size & Overlap

### Concept, Intuition, Why It Exists

- Chunk size and overlap are the two knobs that control the trade-off between coherence and context-loss at chunk boundaries.
- **Chunk size** is straightforward in concept but has competing pressures in practice: too large, and a chunk drifts back toward the blurry-embedding problem from the previous topic; too small, and a chunk loses the surrounding context that gives it meaning — a sentence like "this also applies to senior citizens" is meaningless on its own if the chunk before it (which defined what "this" refers to) got cut off.
- **Overlap** exists specifically to soften the chunk-boundary problem: instead of chunk N ending exactly where chunk N+1 begins, the last portion of chunk N is repeated as the first portion of chunk N+1. This means a sentence or idea that happens to fall near a boundary isn't fully lost to one side — it appears, at least partially, in both neighboring chunks, so whichever one gets retrieved still carries enough context to be useful.
- **This project's `chunk_text()` currently has zero overlap** — a real, named gap, not a stylistic choice. Every chunk boundary is a hard cut with nothing carried forward, meaning any idea that happens to span exactly across two chunks gets fragmented in both directions with no chunk containing the complete thought.

### Internal Working

- With overlap, chunking no longer advances by exactly `chunk_size` characters/sentences per step — it advances by `chunk_size - overlap`, so each new chunk re-includes the trailing `overlap` portion of the previous one.
- Concretely, for sentence-aware chunking: instead of starting the next chunk at the sentence immediately after where the last chunk ended, the next chunk starts a few sentences *before* that boundary — repeating the last `overlap` sentences (or characters) of the previous chunk before continuing with new content.
- This means the same piece of text near a boundary can now appear in two chunks' embeddings — a deliberate redundancy traded for safety against boundary-cut meaning loss.

### How It's Implemented In This Project

- The fix layers an `overlap` parameter onto the existing sentence-aware logic: after filling a chunk up to `chunk_size`, instead of starting completely fresh, the next chunk's starting point steps back by `overlap` sentences (or characters) before continuing forward — see the code below.

### Real-World Issues, Edge Cases, Debugging

- **Zero overlap's actual failure mode**: a policy clause like "premature withdrawal incurs a 1% penalty. This does not apply if the FD is closed due to the death of the depositor." — if the chunk boundary happens to land between these two sentences, one chunk says "1% penalty" with no exception mentioned, and the other says "this does not apply" with no idea what "this" refers to. A query about the death-of-depositor exception could retrieve either half and give an incomplete or actively misleading answer.
- **Overlap isn't free**: more overlap means more redundant text re-embedded and re-stored, directly increasing embedding cost and vector store size for the same source corpus. There's a real budget trade-off, not just a quality dial to crank up.
- **Choosing chunk size is itself content-dependent**: a dense policy clause needs a different chunk size than a conversational FAQ answer — a single fixed chunk size across all sources is a simplification, not a guarantee of correctness for every content type.
- **Debugging boundary loss**: if a known fact is in the source document but retrieval consistently misses it, check whether that fact happens to sit exactly at a chunk boundary before assuming the embedding or retrieval ranking is at fault — this is a very common, very specific root cause.

### Design Decisions & Trade-offs

- Overlap measured in sentences vs. characters: sentence-based overlap (carry forward the last N sentences) keeps boundaries coherent the same way sentence-aware chunking does in the first place; character-based overlap is simpler to implement but can re-cut mid-sentence at the new boundary, partially undermining the reason sentence-aware chunking was chosen.
- Fixed overlap value vs. content-aware overlap: a constant number of sentences/characters is simple and predictable; a smarter approach could vary overlap based on detected sentence boundary density or topic shifts, but adds real complexity for a marginal gain at this project's current scale.

### Alternatives & When To Use Each

- Zero overlap — acceptable only when source content has very low information density per sentence and ideas are unlikely to span exactly across a chunk boundary; risky as a default.
- Fixed sentence-count overlap (the fix here) — good general-purpose default, cheap to implement, directly addresses the boundary-cut problem.
- Fixed character-count overlap — simpler to implement than sentence-counting, but can still cut mid-sentence at the new chunk's start.
- Sliding-window overlap with no discrete chunk boundaries at all — maximizes boundary safety, but multiplies storage/embedding cost significantly since nearly all content gets embedded more than once; rarely worth it outside specialized high-recall use cases.

### Common Mistakes & Production Failures

- Shipping a chunker with zero overlap and discovering boundary-cut information loss only after a customer-facing answer misses an exception clause that was technically in the source document.
- Setting overlap too large relative to chunk size, causing massive redundancy where most of each chunk is just a repeat of its neighbor, inflating cost without meaningfully improving retrieval.
- Picking one global chunk size for every source type without checking whether dense policy text and conversational FAQ content actually need different sizes.

### Lead-Level Interview Questions

**Q: Why does zero overlap matter in practice, given that sentence-aware chunking already avoids cutting mid-sentence?**
A: Sentence-aware chunking guarantees individual sentences aren't broken, but says nothing about *related* sentences that happen to straddle a chunk boundary — an exception clause referencing "this" from the sentence before it can still be split across two chunks, each missing half the meaning. Overlap exists specifically to soften that cross-sentence boundary loss, which sentence-awareness alone doesn't solve.

**Q: How would you decide how much overlap to use, rather than picking an arbitrary number?**
A: Tie it to the typical length of a complete idea in your content — if exception clauses or related statements in this corpus typically span two to three sentences, overlap should cover at least that many sentences so a boundary cut never fully separates a complete thought. It should be validated against real retrieval failures (facts known to be in the source but missed at query time), not chosen purely theoretically.

**Q: What's the actual cost of adding overlap, and how would you reason about whether it's worth it?**
A: Overlap means some text gets embedded and stored more than once, increasing embedding calls and vector store size proportionally to the overlap fraction. It's worth it when boundary-cut information loss is a measured, real problem — confirmed by retrieval misses traceable to chunk boundaries — not applied reflexively as a default without checking whether the corpus actually has content dense enough to need it.

### Hidden Concepts Worth Knowing

- **The boundary-cut problem is a specific instance of a general windowing trade-off**: any system that processes a continuous stream in fixed windows (time-series analysis, audio processing, log analysis) faces the same tension — overlap trades redundancy for safety against information that happens to span a window edge.
- **Overlap doesn't fully solve the problem, it just reduces its frequency**: a sufficiently long related passage can still span beyond even a generous overlap window. Overlap shrinks the failure rate, it doesn't eliminate the failure mode entirely — worth stating honestly rather than treating it as a complete fix.

### Revision Summary

> Chunk size and overlap jointly control the coherence-vs-context-loss trade-off at chunk boundaries. This project's `chunk_text()` currently has zero overlap — a real gap where related ideas spanning a chunk boundary can be split with neither resulting chunk containing the complete thought. Adding overlap (carrying forward the last N sentences into the next chunk) directly addresses this, at the cost of some redundant embedding/storage — a trade that should be sized against real content, not picked arbitrarily.

In [1]:
"""
chunker.py
-----------
Sentence-aware chunking with configurable overlap.

Splits a Document's page_content into smaller Documents, never cutting
mid-sentence, and -- unlike a zero-overlap chunker -- carries forward the
last `overlap` sentences of each chunk into the start of the next one, so
an idea spanning a chunk boundary isn't fully lost to one side.
"""

import re
from document import make_document

SENTENCE_SPLIT_RE = re.compile(r"(?<=[.!?])\s+")


def split_sentences(text: str) -> list:
    """Naive sentence splitter -- splits after ./!/? followed by whitespace.
    Good enough for structured policy/FAQ text; a real NLP sentence
    tokenizer (e.g. spaCy) would handle abbreviations more robustly."""
    sentences = SENTENCE_SPLIT_RE.split(text.strip())
    return [s.strip() for s in sentences if s.strip()]


def chunk_text(text: str, chunk_size: int = 500, overlap: int = 1) -> list:
    """Sentence-aware chunking with overlap.

    chunk_size : max characters per chunk (soft limit -- a chunk stops
                 adding sentences once it would exceed this length).
    overlap    : number of trailing sentences from the previous chunk to
                 repeat at the start of the next chunk. 0 = no overlap
                 (the old, gap-having behavior).

    IMPORTANT: overlap is automatically capped so it can never consume an
    entire completed chunk. If it did, carrying back "the last N sentences"
    would just rebuild the identical chunk every time and the loop would
    never make progress -- a real infinite-loop bug, not just slowness.
    """
    sentences = split_sentences(text)
    if not sentences:
        return []

    chunks = []
    current = []
    current_len = 0
    i = 0

    while i < len(sentences):
        sentence = sentences[i]
        if current_len + len(sentence) > chunk_size and current:
            chunks.append(" ".join(current))

            # Cap overlap so at least ONE sentence is always dropped --
            # this is what guarantees forward progress.
            safe_overlap = min(overlap, len(current) - 1) if overlap > 0 else 0
            carry_back = current[-safe_overlap:] if safe_overlap > 0 else []

            current = list(carry_back)
            current_len = sum(len(s) for s in current)
            continue  # re-evaluate the same sentence against the new chunk
        current.append(sentence)
        current_len += len(sentence)
        i += 1

    if current:
        chunks.append(" ".join(current))

    return chunks


def chunk_document(doc: dict, chunk_size: int = 500, overlap: int = 1) -> list:
    """Splits one Document into several smaller Documents, preserving and
    extending metadata with a chunk index so each piece is still traceable
    back to its source/page/row."""
    pieces = chunk_text(doc["page_content"], chunk_size=chunk_size, overlap=overlap)
    result = []
    for idx, piece in enumerate(pieces):
        meta = dict(doc["metadata"])
        meta["chunk_index"] = idx
        result.append(make_document(page_content=piece, metadata=meta))
    return result


if __name__ == "__main__":
    sample = (
        "Premature withdrawal incurs a 1 percent penalty on the applicable rate. "
        "This does not apply if the FD is closed due to the death of the depositor. "
        "In such cases, the full contracted interest rate is paid up to the date of closure. "
        "Senior citizens receive an additional 0.5 percent interest on all tenures. "
        "This additional rate applies only to resident senior citizens aged 60 and above."
    )

    # chunk_size bumped to 180 so each chunk can actually fit 2 sentences --
    # at 120, no chunk could ever hold more than 1 sentence, so overlap had
    # nothing to carry forward and silently did nothing.
    CHUNK_SIZE = 180

    print("--- Zero overlap (the old gap) ---")
    for i, c in enumerate(chunk_text(sample, chunk_size=CHUNK_SIZE, overlap=0)):
        print(f"  Chunk {i}: {c}")

    print("\n--- With overlap=1 (the fix) ---")
    for i, c in enumerate(chunk_text(sample, chunk_size=CHUNK_SIZE, overlap=1)):
        print(f"  Chunk {i}: {c}")

    print("\n--- As Documents (chunk_document) ---")
    doc = make_document(sample, {"source": "fd_policy_demo.txt", "page": 1})
    for d in chunk_document(doc, chunk_size=CHUNK_SIZE, overlap=1):
        print(f"  {d['metadata']}: {d['page_content'][:60]!r}...")

--- Zero overlap (the old gap) ---
  Chunk 0: Premature withdrawal incurs a 1 percent penalty on the applicable rate. This does not apply if the FD is closed due to the death of the depositor.
  Chunk 1: In such cases, the full contracted interest rate is paid up to the date of closure. Senior citizens receive an additional 0.5 percent interest on all tenures.
  Chunk 2: This additional rate applies only to resident senior citizens aged 60 and above.

--- With overlap=1 (the fix) ---
  Chunk 0: Premature withdrawal incurs a 1 percent penalty on the applicable rate. This does not apply if the FD is closed due to the death of the depositor.
  Chunk 1: This does not apply if the FD is closed due to the death of the depositor. In such cases, the full contracted interest rate is paid up to the date of closure.
  Chunk 2: In such cases, the full contracted interest rate is paid up to the date of closure. Senior citizens receive an additional 0.5 percent interest on all tenures.
  Chunk 3: Se